# Exploration du dataset DVF (demandes de valeur foncière) de data.gouv

On commences par créer une constante pour le dataset DVF brut, 

In [2]:
DATASET_URL = "https://www.data.gouv.fr/api/1/datasets/r/902db087-b0eb-4cbb-a968-0b499bde5bc4"

In [3]:
import httpx

r = httpx.head(DATASET_URL, follow_redirects=True)

for k,v in r.headers.items():
    print(f"{k}: {v}")

server: nginx
date: Tue, 02 Jun 2026 19:43:51 GMT
content-type: application/zip
content-length: 69665132
last-modified: Sun, 05 Apr 2026 00:23:28 GMT
etag: "69d1ab80-427016c"
expires: Thu, 02 Jul 2026 19:43:51 GMT
cache-control: max-age=2592000, public
content-disposition: attachment
pragma: public
x-content-type-options: nosniff
x-xss-protection: 1; mode=block
x-frame-options: SAMEORIGIN
access-control-allow-origin: *
access-control-allow-methods: GET, OPTIONS
access-control-allow-headers: Origin,Authorization,Accept,DNT,User-Agent,If-Modified-Since,Cache-Control,Content-Type,Range,sentry-trace,baggage
access-control-allow-credentials: true
accept-ranges: bytes
strict-transport-security: max-age=86400; includeSubDomains; preload;


On récupéres les headers d'une réponse potentielle du serveur avec la méthode HEAD, cela nous apprends qu'il s'agit d'une archive ZIP et la taille du fichier, en octet ce n'est pas trés lisible on fait donc une conversion en mégaoctets.

In [4]:

length_mb = round(int(r.headers["content-length"]) / 1024**2, 2)
print(f"\nContent-Length: {length_mb} Mo")


Content-Length: 66.44 Mo


On télécharges le ZIP en local, la structure standard d'un projet Kedro possèdes des dossiers pour stocker nos données de travail. 

On va ensuite lire l'archive et extraire son contenu toujours dans les dossier 01_raw pour les données brutes.

In [5]:
import zipfile

CHUNK_SIZE = 1024 * 1024 # 1 Mo
DIR_PATH = "../data/01_raw/"
ZIP_FILE_PATH = DIR_PATH + "dvf.zip"


with httpx.stream("GET", DATASET_URL, follow_redirects=True) as response:
    with open(ZIP_FILE_PATH, "wb") as file:
        for chunk in response.iter_bytes(CHUNK_SIZE):
            file.write(chunk)


with zipfile.ZipFile(ZIP_FILE_PATH, mode = "r") as zip:
    if zip.testzip():
        print("Zipfile is corrupted")
 
    namelist = zip.namelist()

    zip.extractall(DIR_PATH)



On s'apperçoit que l'archive ne contient qu'un seul fichier texte. On aura besoin de savoir le format des données à l'intérieur.

On va lire et afficher les 1000 premiers charactères pour voir ce qu'il y'a dedans.

In [6]:
DVF_FILE_PATH = DIR_PATH + namelist[0]

with open(DVF_FILE_PATH, mode = "r", encoding="utf-8") as dvf:
    chunk = dvf.read(1000)

print(chunk)

Identifiant de document|Reference document|1 Articles CGI|2 Articles CGI|3 Articles CGI|4 Articles CGI|5 Articles CGI|No disposition|Date mutation|Nature mutation|Valeur fonciere|No voie|B/T/Q|Type de voie|Code voie|Voie|Code postal|Commune|Code departement|Code commune|Prefixe de section|Section|No plan|No Volume|1er lot|Surface Carrez du 1er lot|2eme lot|Surface Carrez du 2eme lot|3eme lot|Surface Carrez du 3eme lot|4eme lot|Surface Carrez du 4eme lot|5eme lot|Surface Carrez du 5eme lot|Nombre de lots|Code type local|Type local|Identifiant local|Surface reelle bati|Nombre pieces principales|Nature culture|Nature culture speciale|Surface terrain
|||||||000001|07/01/2025|Vente|468000,00||||B078|FARGES|1550|FARGES|01|158||B|815||||||||||||0||||||J||78
|||||||000001|07/01/2025|Vente|468000,00|454||RUE|0090|DE LA REPUBLIQUE|1550|FARGES|01|158||B|910||||||||||||0|3|Dépendance||0|0|S||133
|||||||000001|07/01/2025|Vente|468000,00|454||RUE|0090|DE LA REPUBLIQUE|1550|FARGES|01|158||B|910||||||

Il semblerait qu'il s'agisse d'un fichier CSV utilisant des pipes '|' comme séparateurs donc on va le traiter comme tel.

Comme on à vu plus tot le fichier zippé avait une taille de 66.44 Mo on veut aussi connaitre sa taille décompréssée avant de le lire en mémoire.

In [7]:
import os

file_size = os.path.getsize(DVF_FILE_PATH)
file_size_mb = round(file_size / 1024**2, 2)

print(f"The decompressed DVF file size is: {file_size_mb} Mb")

The decompressed DVF file size is: 473.66 Mb


Okay donc 470 Mo sur disque, il ne faut pas oublier que le chargmeent en mémoire sous forme d'objet gonfle la taille de 3x à 10x.

On va commencer par charger 100 lignes pour l'exploration.

In [8]:
import pandas as pd

valeurs_foncieres = pd.read_csv(DVF_FILE_PATH, sep="|", nrows=100)


In [9]:
valeurs_foncieres.dtypes


Identifiant de document       float64
Reference document            float64
1 Articles CGI                float64
2 Articles CGI                float64
3 Articles CGI                float64
4 Articles CGI                float64
5 Articles CGI                float64
No disposition                  int64
Date mutation                     str
Nature mutation                   str
Valeur fonciere                   str
No voie                       float64
B/T/Q                             str
Type de voie                      str
Code voie                         str
Voie                              str
Code postal                     int64
Commune                           str
Code departement                int64
Code commune                    int64
Prefixe de section            float64
Section                           str
No plan                         int64
No Volume                     float64
1er lot                       float64
Surface Carrez du 1er lot         str
2eme lot   

In [10]:
valeurs_foncieres.head()

,Identifiant de document,Reference document,1 Articles CGI,2 Articles CGI,3 Articles CGI,4 Articles CGI,5 Articles CGI,No disposition,Date mutation,Nature mutation,...,Surface Carrez du 5eme lot,Nombre de lots,Code type local,Type local,Identifiant local,Surface reelle bati,Nombre pieces principales,Nature culture,Nature culture speciale,Surface terrain
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,07/01/2025,Vente,...,NaN,0,NaN,NaN,NaN,NaN,NaN,J,NaN,78.0
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,07/01/2025,Vente,...,NaN,0,3.0,Dépendance,NaN,0.0,0.0,S,NaN,133.0
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,07/01/2025,Vente,...,NaN,0,1.0,Maison,NaN,111.0,5.0,S,NaN,133.0
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,06/01/2025,Vente,...,NaN,0,NaN,NaN,NaN,NaN,NaN,S,NaN,46.0
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,06/01/2025,Vente,...,NaN,0,NaN,NaN,NaN,NaN,NaN,J,NaN,17.0


Problème ! On constates qu'on à pas de données géographiques telles quelles, il existes un version améliorée du dataset qu'on va utiliser à la place.

https://www.data.gouv.fr/datasets/demandes-de-valeurs-foncieres-geolocalisees

In [11]:
GEO_DVF_URL = "https://www.data.gouv.fr/api/1/datasets/r/316795eb-a3fa-465d-b058-38ef8579da11"

In [12]:

res = httpx.head(GEO_DVF_URL, follow_redirects=True)

for k,v in res.headers.items():
    print(f"{k}: {v}")

server: nginx
date: Tue, 02 Jun 2026 19:44:07 GMT
content-type: text/html
connection: keep-alive
vary: Accept-Encoding
strict-transport-security: max-age=86400; includeSubDomains; preload;
content-encoding: gzip


Pas d'information concernant la taille du fichier mais on apprends qu'il s'agit d'un fichier GZIP c'est à dire un unique fichier compréssé plutot qu'une archive.

La documentation du dataset nous indiques qu'il s'agit d'un CSV avec séparateur virgule et encodage UTF-8

In [ ]:
import gzip

GZIP_FILE_PATH = DIR_PATH + "geo_dvf.csv.gz"

# On télécharge l'archive GZIP en local
with httpx.stream("GET", GEO_DVF_URL, follow_redirects=True) as response:
    with open(GZIP_FILE_PATH, "wb") as file:
        for chunk in response.iter_bytes(CHUNK_SIZE):
            file.write(chunk)

# gzip = un unique fichier compressé : on lit directement le .gz décompressé à la volée.
# Mode "rt" (text) + encoding pour obtenir une str et non des bytes.
with gzip.open(GZIP_FILE_PATH, mode="rt", encoding="utf-8") as csv_gz:
    print(csv_gz.read(1000))